<a href="https://colab.research.google.com/github/vivek2096kumar-singh/PERSONAL_HEALTH_ASSISTANT_COLLAB/blob/main/Personal_Health_Assistant_Week3to4_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🩺 Personal Health Assistant — Healthcare Monitoring AI Agent
### Track A (Essential) — Weeks 1-4 Combined — Google Colab Version

Run each cell **top to bottom**. This notebook now covers:

**Week 1-2 (Foundation & Quick Win):**
- Medication tracker (add, mark taken/missed, adherence chart)
- Fitness logger (steps, calories, sleep, water)
- Health info lookup (MedlinePlus API + offline fallback)

**Week 3-4 (Core Healthcare Agent Architecture) — NEW:**
- 🔗 Medication interaction checking
- 📄 Simple health report generation
- 🔄 Multi-format data handling (JSON, CSV, XML import/export)
- 🎯 Health goal setting and progress tracking

**Author:** Rohit Yadav | AI & Data Science, GITM Gurugram


## Step 1: Install dependencies

In [1]:
!pip install gradio pandas requests matplotlib -q
print("Installed successfully!")


Installed successfully!


## Step 2: Imports

In [2]:
import sqlite3
import re
import json
import csv
import io
import xml.etree.ElementTree as ET
from xml.dom import minidom
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, datetime
import gradio as gr

print("Imports ready.")


Imports ready.


## Step 3: Set up the database

Creates a local SQLite file `health_assistant.db`. Adds a new `goals` table
for Week 3-4 goal tracking on top of the Week 1-2 schema.

In [3]:
DB_PATH = "health_assistant.db"

def get_connection():
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS medications (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            dosage TEXT,
            schedule_time TEXT NOT NULL,
            notes TEXT,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS medication_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            medication_id INTEGER NOT NULL,
            log_date TEXT NOT NULL,
            status TEXT NOT NULL CHECK(status IN ('taken', 'missed')),
            logged_at TEXT DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (medication_id) REFERENCES medications (id)
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS health_metrics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            entry_date TEXT NOT NULL UNIQUE,
            steps INTEGER DEFAULT 0,
            calories INTEGER DEFAULT 0,
            sleep_hours REAL DEFAULT 0,
            water_glasses INTEGER DEFAULT 0
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS goals (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            goal_type TEXT NOT NULL,
            target_value REAL NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

init_db()
print("Database ready:", DB_PATH)


Database ready: health_assistant.db


## Step 4: Medication tracker functions (Week 1-2)

In [4]:
def add_medication(name, dosage, schedule_time, notes=""):
    conn = get_connection()
    conn.execute(
        "INSERT INTO medications (name, dosage, schedule_time, notes) VALUES (?, ?, ?, ?)",
        (name, dosage, schedule_time, notes),
    )
    conn.commit()
    conn.close()

def get_medications():
    conn = get_connection()
    rows = conn.execute("SELECT * FROM medications ORDER BY schedule_time ASC").fetchall()
    conn.close()
    return rows

def log_medication(med_id, status, log_date=None):
    log_date = log_date or str(date.today())
    conn = get_connection()
    existing = conn.execute(
        "SELECT id FROM medication_logs WHERE medication_id = ? AND log_date = ?",
        (med_id, log_date),
    ).fetchone()
    if existing:
        conn.execute(
            "UPDATE medication_logs SET status = ?, logged_at = ? WHERE id = ?",
            (status, datetime.now().isoformat(), existing["id"]),
        )
    else:
        conn.execute(
            "INSERT INTO medication_logs (medication_id, log_date, status) VALUES (?, ?, ?)",
            (med_id, log_date, status),
        )
    conn.commit()
    conn.close()

def get_today_logs(log_date=None):
    log_date = log_date or str(date.today())
    conn = get_connection()
    rows = conn.execute("SELECT * FROM medication_logs WHERE log_date = ?", (log_date,)).fetchall()
    conn.close()
    return {row["medication_id"]: row["status"] for row in rows}

def get_adherence_stats(days=7):
    conn = get_connection()
    rows = conn.execute(
        """SELECT log_date, status, COUNT(*) as cnt FROM medication_logs
           WHERE log_date >= date('now', ?) GROUP BY log_date, status ORDER BY log_date ASC""",
        (f"-{days} days",),
    ).fetchall()
    conn.close()
    return rows

print("Medication functions ready.")


Medication functions ready.


## Step 5: Fitness tracker functions (Week 1-2)

In [5]:
def upsert_health_metric(entry_date, steps, calories, sleep_hours, water_glasses):
    conn = get_connection()
    existing = conn.execute("SELECT id FROM health_metrics WHERE entry_date = ?", (entry_date,)).fetchone()
    if existing:
        conn.execute(
            """UPDATE health_metrics SET steps=?, calories=?, sleep_hours=?, water_glasses=?
               WHERE entry_date=?""",
            (steps, calories, sleep_hours, water_glasses, entry_date),
        )
    else:
        conn.execute(
            """INSERT INTO health_metrics (entry_date, steps, calories, sleep_hours, water_glasses)
               VALUES (?, ?, ?, ?, ?)""",
            (entry_date, steps, calories, sleep_hours, water_glasses),
        )
    conn.commit()
    conn.close()

def get_health_metrics(days=14):
    conn = get_connection()
    rows = conn.execute(
        "SELECT * FROM health_metrics WHERE entry_date >= date('now', ?) ORDER BY entry_date ASC",
        (f"-{days} days",),
    ).fetchall()
    conn.close()
    return rows

print("Fitness functions ready.")


Fitness functions ready.


## Step 6: Health information lookup (Week 1-2)

In [6]:
MEDLINEPLUS_URL = "https://wsearch.nlm.nih.gov/ws/query"

FALLBACK_INFO = {
    "fever": "A fever is a temporary rise in body temperature, often due to infection. "
             "Stay hydrated and rest; seek medical care if it exceeds 103°F (39.4°C) or lasts more than 3 days.",
    "headache": "Common causes include stress, dehydration, and lack of sleep. Rest, hydration, "
                "and over-the-counter pain relief usually help. See a doctor for sudden severe or recurring headaches.",
    "cold": "The common cold is a viral infection of the nose and throat. Rest, fluids, and time "
            "are the main treatments; most colds resolve in 7-10 days.",
    "diabetes": "Diabetes is a condition where blood sugar levels are too high. Management includes "
                "diet, exercise, monitoring, and medication as prescribed by a doctor.",
    "blood pressure": "Blood pressure measures the force of blood against artery walls. Normal is "
                       "roughly below 120/80 mmHg. Consistently high readings should be discussed with a doctor.",
}

def search_health_info(query):
    query = (query or "").strip()
    if not query:
        return "Please enter a health topic to search."
    try:
        params = {"db": "healthTopics", "term": query, "retmax": 1}
        resp = requests.get(MEDLINEPLUS_URL, params=params, timeout=5)
        resp.raise_for_status()
        text = resp.text
        if "<document" in text:
            snippet_match = re.search(r"<content name=\"snippet\">(.*?)</content>", text, re.S)
            if snippet_match:
                snippet = re.sub("<.*?>", "", snippet_match.group(1)).strip()
                if snippet:
                    return f"**Source: MedlinePlus**\n\n{snippet}"
    except Exception:
        pass

    key = query.lower()
    for fkey, info in FALLBACK_INFO.items():
        if fkey in key or key in fkey:
            return f"**Source: Offline reference**\n\n{info}"

    return ("No information found for this topic. Try a common term like 'fever', "
            "'headache', 'cold', 'diabetes', or 'blood pressure', or consult a medical professional.")

print("Health info lookup ready.")


Health info lookup ready.


## Step 7: Medication interaction checker (Week 3-4 — NEW)

A rule-based checker against a curated list of well-known interaction pairs.
This is for **educational purposes only** — it is not a substitute for a
pharmacist or doctor's review, and does not cover all possible interactions.

In [7]:
# Curated list of commonly cited drug interaction pairs (educational only).
# Format: frozenset({drug_a, drug_b}) -> (severity, message)
KNOWN_INTERACTIONS = {
    frozenset({"warfarin", "aspirin"}): ("High", "Increased risk of bleeding when combined."),
    frozenset({"warfarin", "ibuprofen"}): ("High", "NSAIDs like ibuprofen can increase bleeding risk with warfarin."),
    frozenset({"metformin", "alcohol"}): ("Moderate", "Alcohol can increase risk of lactic acidosis with metformin."),
    frozenset({"aspirin", "ibuprofen"}): ("Moderate", "Combining NSAIDs may increase GI bleeding/ulcer risk."),
    frozenset({"lisinopril", "ibuprofen"}): ("Moderate", "NSAIDs may reduce the blood-pressure-lowering effect of ACE inhibitors."),
    frozenset({"simvastatin", "grapefruit"}): ("Moderate", "Grapefruit can raise statin levels, increasing side-effect risk."),
    frozenset({"levothyroxine", "calcium"}): ("Low", "Calcium supplements can reduce levothyroxine absorption if taken together."),
    frozenset({"sertraline", "ibuprofen"}): ("Moderate", "Combining SSRIs with NSAIDs may increase bleeding risk."),
}

def check_medication_interactions():
    """Checks all pairwise combinations of the user's current medication list."""
    meds = get_medications()
    names = [m["name"].strip().lower() for m in meds]
    warnings = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            pair = frozenset({names[i], names[j]})
            if pair in KNOWN_INTERACTIONS:
                severity, message = KNOWN_INTERACTIONS[pair]
                warnings.append(f"⚠️ **{severity} risk** — {names[i].title()} + {names[j].title()}: {message}")
    if not warnings:
        return ("✅ No known interactions found among your current medications "
                "(based on our reference list). This is not a complete medical check — "
                "always confirm with a pharmacist or doctor.")
    disclaimer = ("\n\n⚠️ *This check covers a limited reference list of common interactions "
                  "and is for educational purposes only. Always consult a pharmacist or doctor.*")
    return "\n\n".join(warnings) + disclaimer

print("Interaction checker ready.")
print(check_medication_interactions())


Interaction checker ready.
✅ No known interactions found among your current medications (based on our reference list). This is not a complete medical check — always confirm with a pharmacist or doctor.


## Step 8: Health goal setting & progress tracking (Week 3-4 — NEW)

In [8]:
GOAL_TYPES = ["steps", "sleep_hours", "water_glasses", "calories"]

def set_goal(goal_type, target_value):
    conn = get_connection()
    existing = conn.execute("SELECT id FROM goals WHERE goal_type = ?", (goal_type,)).fetchone()
    if existing:
        conn.execute("UPDATE goals SET target_value = ? WHERE goal_type = ?", (target_value, goal_type))
    else:
        conn.execute("INSERT INTO goals (goal_type, target_value) VALUES (?, ?)", (goal_type, target_value))
    conn.commit()
    conn.close()

def get_goals():
    conn = get_connection()
    rows = conn.execute("SELECT * FROM goals").fetchall()
    conn.close()
    return {row["goal_type"]: row["target_value"] for row in rows}

def get_goal_progress():
    goals = get_goals()
    if not goals:
        return "No goals set yet. Set a goal below to start tracking progress."
    metrics = get_health_metrics(days=1)
    today = next((dict(r) for r in metrics if r["entry_date"] == str(date.today())), None)
    if not today:
        return "No fitness data logged today yet. Log today's entry in the Fitness Tracker tab first."

    lines = []
    for goal_type, target in goals.items():
        actual = today.get(goal_type, 0)
        pct = min(100, round((actual / target) * 100, 1)) if target else 0
        bar = "🟩" * int(pct // 10) + "⬜" * (10 - int(pct // 10))
        lines.append(f"**{goal_type.replace('_', ' ').title()}**: {actual}/{target} — {bar} {pct}%")
    return "\n\n".join(lines)

print("Goal tracking ready.")


Goal tracking ready.


## Step 9: Health report generation (Week 3-4 — NEW)

Generates a simple end-to-end health summary report combining medications,
adherence, fitness trends, and goal progress.

In [9]:
def generate_health_report(days=7):
    lines = [f"# Health Report — Last {days} Days", f"_Generated on {date.today().isoformat()}_", ""]

    # Medications section
    meds = get_medications()
    lines.append("## Medications")
    if meds:
        for m in meds:
            lines.append(f"- **{m['name']}** ({m['dosage']}) at {m['schedule_time']}")
    else:
        lines.append("- No medications on record.")

    # Adherence section
    stats = get_adherence_stats(days)
    lines.append("\n## Medication Adherence")
    if stats:
        df = pd.DataFrame([dict(r) for r in stats])
        taken = df[df["status"] == "taken"]["cnt"].sum()
        missed = df[df["status"] == "missed"]["cnt"].sum()
        total = taken + missed
        rate = round((taken / total) * 100, 1) if total else 0
        lines.append(f"- Taken: {taken} | Missed: {missed} | Adherence rate: {rate}%")
    else:
        lines.append("- No adherence data logged yet.")

    # Fitness section
    metrics = get_health_metrics(days)
    lines.append("\n## Fitness Summary")
    if metrics:
        df = pd.DataFrame([dict(r) for r in metrics])
        lines.append(f"- Avg steps/day: {df['steps'].mean():.0f}")
        lines.append(f"- Avg calories burned/day: {df['calories'].mean():.0f}")
        lines.append(f"- Avg sleep/night: {df['sleep_hours'].mean():.1f} hrs")
        lines.append(f"- Avg water intake/day: {df['water_glasses'].mean():.1f} glasses")
    else:
        lines.append("- No fitness data logged yet.")

    # Goals section
    lines.append("\n## Goal Progress")
    lines.append(get_goal_progress())

    # Interaction check
    lines.append("\n## Medication Interaction Check")
    lines.append(check_medication_interactions())

    return "\n".join(lines)

print("Report generator ready.")


Report generator ready.


## Step 10: Multi-format data handling — JSON, CSV, XML (Week 3-4 — NEW)

Export your fitness metrics to JSON, CSV, or XML, and import fitness data
from any of those three formats.

In [10]:
def export_metrics(fmt="json", days=30):
    rows = [dict(r) for r in get_health_metrics(days)]
    if not rows:
        return None, "No data to export."

    if fmt == "json":
        content = json.dumps(rows, indent=2)
        filename = "health_metrics_export.json"
    elif fmt == "csv":
        buf = io.StringIO()
        writer = csv.DictWriter(buf, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
        content = buf.getvalue()
        filename = "health_metrics_export.csv"
    elif fmt == "xml":
        root = ET.Element("health_metrics")
        for row in rows:
            entry = ET.SubElement(root, "entry")
            for k, v in row.items():
                child = ET.SubElement(entry, k)
                child.text = str(v)
        rough = ET.tostring(root, encoding="unicode")
        content = minidom.parseString(rough).toprettyxml(indent="  ")
        filename = "health_metrics_export.xml"
    else:
        return None, "Unsupported format."

    with open(filename, "w") as f:
        f.write(content)
    return filename, f"Exported {len(rows)} records to {filename}"


def import_metrics(file_obj):
    if file_obj is None:
        return "No file uploaded."
    path = file_obj.name if hasattr(file_obj, "name") else file_obj
    ext = path.split(".")[-1].lower()
    count = 0
    try:
        if ext == "json":
            with open(path) as f:
                rows = json.load(f)
        elif ext == "csv":
            with open(path) as f:
                rows = list(csv.DictReader(f))
        elif ext == "xml":
            tree = ET.parse(path)
            rows = []
            for entry in tree.getroot().findall("entry"):
                rows.append({child.tag: child.text for child in entry})
        else:
            return f"Unsupported file type: .{ext}. Use JSON, CSV, or XML."

        for row in rows:
            upsert_health_metric(
                row["entry_date"],
                int(float(row.get("steps", 0))),
                int(float(row.get("calories", 0))),
                float(row.get("sleep_hours", 0)),
                int(float(row.get("water_glasses", 0))),
            )
            count += 1
        return f"✅ Imported {count} records from {ext.upper()} file."
    except Exception as e:
        return f"⚠️ Import failed: {e}"

print("Import/export functions ready.")


Import/export functions ready.


## Step 11: Build the Gradio interface

Combines Week 1-2 tabs (Medications, Fitness Tracker, Health Info Lookup)
with new Week 3-4 tabs (Interaction Checker, Goals, Reports, Data Import/Export).

In [11]:
def ui_add_medication(name, dosage, hour, notes):
    if not name.strip():
        return "⚠️ Please enter a medication name.", ui_list_medications()
    add_medication(name.strip(), dosage.strip(), hour.strip(), notes.strip())
    return f"✅ Added {name}!", ui_list_medications()

def ui_list_medications():
    meds = get_medications()
    logs = get_today_logs()
    if not meds:
        return "No medications added yet."
    lines = []
    for m in meds:
        status = logs.get(m["id"], "pending")
        icon = "✅" if status == "taken" else ("❌" if status == "missed" else "⏳")
        lines.append(f"{icon} **{m['name']}** ({m['dosage']}) — {m['schedule_time']} — ID {m['id']} — {status}")
    return "\n\n".join(lines)

def ui_mark_status(med_id, status):
    try:
        med_id = int(med_id)
    except ValueError:
        return "⚠️ Enter a valid medication ID from the list above.", ui_list_medications()
    log_medication(med_id, status)
    return f"Marked as {status}.", ui_list_medications()

def ui_adherence_chart():
    stats = get_adherence_stats(7)
    if not stats:
        return None
    df = pd.DataFrame([dict(r) for r in stats])
    pivot = df.pivot_table(index="log_date", columns="status", values="cnt", fill_value=0)
    fig, ax = plt.subplots(figsize=(6, 3))
    pivot.plot(kind="bar", ax=ax)
    ax.set_title("7-Day Medication Adherence")
    plt.tight_layout()
    return fig

def ui_save_fitness(steps, calories, sleep_hours, water):
    today_str = str(date.today())
    upsert_health_metric(today_str, int(steps), int(calories), float(sleep_hours), int(water))
    return "✅ Saved today's fitness data!", ui_fitness_chart()

def ui_fitness_chart():
    rows = get_health_metrics(14)
    if not rows:
        return None
    df = pd.DataFrame([dict(r) for r in rows])
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(df["entry_date"], df["steps"], marker="o", label="Steps")
    ax.set_title("Steps — Last 14 Days")
    plt.xticks(rotation=45)
    plt.tight_layout()
    return fig

def ui_set_goal(goal_type, target_value):
    if not target_value or target_value <= 0:
        return "⚠️ Enter a valid target value.", get_goal_progress()
    set_goal(goal_type, target_value)
    return f"✅ Goal set: {goal_type.replace('_',' ').title()} = {target_value}", get_goal_progress()

def ui_export(fmt):
    filename, message = export_metrics(fmt)
    return filename, message

with gr.Blocks(title="Personal Health Assistant") as demo:
    gr.Markdown("# 🩺 Personal Health Assistant\n*Weeks 1-4 — Healthcare Monitoring AI Agent*")

    with gr.Tab("Medications"):
        gr.Markdown("### Add a medication")
        with gr.Row():
            name_in = gr.Textbox(label="Medication name")
            dosage_in = gr.Textbox(label="Dosage (e.g. 500mg)")
            time_in = gr.Textbox(label="Reminder time (HH:MM)", placeholder="08:00")
        notes_in = gr.Textbox(label="Notes (optional)")
        add_btn = gr.Button("Add Medication")
        add_status = gr.Markdown()

        gr.Markdown("### Your medications")
        med_list = gr.Markdown(ui_list_medications())
        refresh_btn = gr.Button("Refresh List")

        gr.Markdown("### Mark today's status")
        with gr.Row():
            med_id_in = gr.Textbox(label="Medication ID (from list above)")
            mark_taken_btn = gr.Button("Mark Taken ✅")
            mark_missed_btn = gr.Button("Mark Missed ❌")
        mark_status = gr.Markdown()

        gr.Markdown("### 7-Day Adherence Chart")
        chart_btn = gr.Button("Show Adherence Chart")
        adherence_plot = gr.Plot()

        add_btn.click(ui_add_medication, [name_in, dosage_in, time_in, notes_in], [add_status, med_list])
        refresh_btn.click(ui_list_medications, [], med_list)
        mark_taken_btn.click(lambda mid: ui_mark_status(mid, "taken"), [med_id_in], [mark_status, med_list])
        mark_missed_btn.click(lambda mid: ui_mark_status(mid, "missed"), [med_id_in], [mark_status, med_list])
        chart_btn.click(ui_adherence_chart, [], adherence_plot)

    with gr.Tab("Fitness Tracker"):
        gr.Markdown("### Log today's activity")
        with gr.Row():
            steps_in = gr.Number(label="Steps", value=0)
            calories_in = gr.Number(label="Calories burned", value=0)
        with gr.Row():
            sleep_in = gr.Number(label="Sleep (hours)", value=0)
            water_in = gr.Number(label="Water (glasses)", value=0)
        save_fit_btn = gr.Button("Save Today's Entry")
        fit_status = gr.Markdown()
        fit_plot = gr.Plot()

        save_fit_btn.click(ui_save_fitness, [steps_in, calories_in, sleep_in, water_in], [fit_status, fit_plot])

    with gr.Tab("Health Info Lookup"):
        gr.Markdown("### Search a health topic\n_e.g. fever, headache, cold, diabetes, blood pressure_")
        query_in = gr.Textbox(label="Health topic")
        search_btn = gr.Button("Search")
        info_out = gr.Markdown()
        search_btn.click(search_health_info, [query_in], info_out)

    with gr.Tab("Interaction Checker (New)"):
        gr.Markdown(
            "### Medication Interaction Check\n"
            "Checks your current medication list against a reference set of "
            "commonly known interactions. **Educational use only.**"
        )
        interaction_btn = gr.Button("Check My Medications")
        interaction_out = gr.Markdown()
        interaction_btn.click(check_medication_interactions, [], interaction_out)

    with gr.Tab("Goals & Progress (New)"):
        gr.Markdown("### Set a health goal")
        with gr.Row():
            goal_type_in = gr.Dropdown(GOAL_TYPES, label="Goal type", value="steps")
            goal_target_in = gr.Number(label="Target value", value=0)
        goal_btn = gr.Button("Set Goal")
        goal_status = gr.Markdown()

        gr.Markdown("### Today's Progress")
        progress_out = gr.Markdown(get_goal_progress())
        refresh_goal_btn = gr.Button("Refresh Progress")

        goal_btn.click(ui_set_goal, [goal_type_in, goal_target_in], [goal_status, progress_out])
        refresh_goal_btn.click(get_goal_progress, [], progress_out)

    with gr.Tab("Health Report (New)"):
        gr.Markdown("### Generate a full health summary report")
        report_days_in = gr.Slider(minimum=1, maximum=30, value=7, step=1, label="Report period (days)")
        report_btn = gr.Button("Generate Report")
        report_out = gr.Markdown()
        report_btn.click(generate_health_report, [report_days_in], report_out)

    with gr.Tab("Data Import/Export (New)"):
        gr.Markdown("### Export fitness data")
        export_fmt_in = gr.Radio(["json", "csv", "xml"], label="Format", value="json")
        export_btn = gr.Button("Export")
        export_file_out = gr.File(label="Download")
        export_msg_out = gr.Markdown()
        export_btn.click(ui_export, [export_fmt_in], [export_file_out, export_msg_out])

        gr.Markdown("### Import fitness data\n_Upload a JSON, CSV, or XML file with columns: "
                    "entry_date, steps, calories, sleep_hours, water_glasses_")
        import_file_in = gr.File(label="Upload file")
        import_btn = gr.Button("Import")
        import_msg_out = gr.Markdown()
        import_btn.click(import_metrics, [import_file_in], import_msg_out)

    gr.Markdown(
        "---\n⚠️ *This tool provides general information only and is not a substitute "
        "for professional medical advice, diagnosis, or treatment. The interaction "
        "checker covers a limited reference list — always consult a pharmacist or doctor.*"
    )

print("Gradio app built.")


Gradio app built.


## Step 12: Launch the app

`share=True` gives you a public link you can use for your demo video.

In [ ]:
demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a9efb3e1d018b4c66a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
